# Трансформеры: многоголовое самовнимание и стратегии декодирования
## Семинар 2

В этом семинаре мы развиваем идеи базового механизма самовнимания и
расширяем их до многоголового самовнимания. Затем рассматриваем стратегии
декодирования для генеративных трансформеров (на примере GPT‑подобной модели)
и экспериментально сравниваем их поведение.

## Цели занятия

После выполнения этого семинара студент должен уметь:

1. Освоить реализацию многоголового самовнимания на основе матричной формы.
2. Понять роль разбиения скрытого пространства на головы и последующей агрегации.
3. Научиться реализовывать и сравнивать различные стратегии декодирования
   (случайное сэмплирование, жадный выбор, top‑k, nucleus‑sampling).
4. Проанализировать влияние гиперпараметров (число голов, k, порог p)
   на качество и разнообразие сгенерированного текста.
5. Сравнить поведение разных стратегий декодирования при фиксированном начальном
   префиксе и пороге случайности.
6. Интерпретировать полученные различия с точки зрения вероятностной модели.

## 3. Теоретическая часть

### 3.1. Многоголовое самовнимание

В многоголовом самовнимании скрытое пространство размерности $D$
разбивается на $H$ голов, каждая из которых работает в меньшем
пространстве размерности $D_h = D / H$. Для каждой головы $h$
вводятся свои параметры:

$$
\Omega_q^{(h)}, \Omega_k^{(h)}, \Omega_v^{(h)} \in \mathbb{R}^{D_h \times D},\quad
\boldsymbol{\beta}_q^{(h)}, \boldsymbol{\beta}_k^{(h)}, \boldsymbol{\beta}_v^{(h)} \in \mathbb{R}^{D_h}.
$$

Каждая голова независимо вычисляет свои запросы, ключи и значения
и порождает выход $X'^{(h)} \in \mathbb{R}^{D_h \times N}$.
Далее результаты всех голов конкатенируются по размерности признаков
и пропускаются через линейный слой $\Omega_c \in \mathbb{R}^{D \times D}$:

$$
\mathrm{MultiHead}(X) = \Omega_c \; [X'^{(1)}; \dots; X'^{(H)}],
$$

где $[\cdot; \dots; \cdot]$ обозначает конкатенацию по размерности признаков.

### 3.2. Стратегии декодирования

Автогрессивная языковая модель задаёт распределение
$p(x_{t+1} \mid x_{1:t})$. На каждом шаге можно выбрать $x_{t+1}$ разными способами:

1. **Жадный выбор (greedy)**: $x = \arg\max_i \pi_i$.
2. **Случайное сэмплирование**: $x \sim \mathrm{Categorical}(\boldsymbol{\pi})$.
3. **Top‑k sampling**: обнуляем все вероятности, кроме $k$ наибольших, и нормируем.
4. **Nucleus sampling (top‑p)**: выбираем минимальное подмножество токенов,
   чья суммарная вероятность $\ge p$, и сэмплируем только из него.

Гипотеза: жадный выбор даёт более детерминированный текст,
в то время как стохастические стратегии увеличивают разнообразие,
но могут ухудшать согласованность.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 4. Практическая часть

### Задача 1. Многоголовое самовнимание (NumPy)

**Постановка задачи.**  
Реализовать многоголовое самовнимание для искусственно сгенерированной
матрицы входов $X \in \mathbb{R}^{D \times N}$, используя $H = 2$
головы и масштабированное скалярное произведение.

**Теоретический минимум.**  
Для каждой головы $h$ вычисляются
$Q^{(h)}, K^{(h)}, V^{(h)} \in \mathbb{R}^{D_h \times N}$,
далее матрица оценок внимания

$$
E^{(h)} = \frac{(Q^{(h)})^\top K^{(h)}}{\sqrt{D_h}},
$$

после чего к ней применяется softmax по строкам (по индексам запросов)
и выходная матрица $X'^{(h)} = V^{(h)} (A^{(h)})^\top$.

Гипотеза: разные головы, даже при случайной инициализации,
могут показывать различные паттерны распределения внимания.

In [ ]:
# Фиксируем сид
np.random.seed(3)

D = 8   # размерность признаков
N = 6   # длина последовательности
H = 2   # число голов
D_h = D // H

# Входная матрица X формы (D, N)
X = np.random.normal(size=(D, N))
print("Форма X:", X.shape)

# Инициализируем параметры голов
np.random.seed(0)

omega_q1 = np.random.normal(size=(D_h, D))
omega_k1 = np.random.normal(size=(D_h, D))
omega_v1 = np.random.normal(size=(D_h, D))

beta_q1 = np.random.normal(size=(D_h, 1))
beta_k1 = np.random.normal(size=(D_h, 1))
beta_v1 = np.random.normal(size=(D_h, 1))

omega_q2 = np.random.normal(size=(D_h, D))
omega_k2 = np.random.normal(size=(D_h, D))
omega_v2 = np.random.normal(size=(D_h, D))

beta_q2 = np.random.normal(size=(D_h, 1))
beta_k2 = np.random.normal(size=(D_h, 1))
beta_v2 = np.random.normal(size=(D_h, 1))

# Общая матрица для объединения голов
omega_c = np.random.normal(size=(D, D))


def softmax_rows(data_in: np.ndarray) -> np.ndarray:
    max_per_row = np.max(data_in, axis=1, keepdims=True)
    exp_values = np.exp(data_in - max_per_row)
    denom = np.sum(exp_values, axis=1, keepdims=True)
    return exp_values / denom


def single_head_scaled_attention(X, omega_q, omega_k, omega_v, beta_q, beta_k, beta_v):
    D_h, D_full = omega_q.shape
    _, N = X.shape
    ones_row = np.ones((1, N))

    Q = omega_q @ X + beta_q @ ones_row  # (D_h, N)
    K = omega_k @ X + beta_k @ ones_row  # (D_h, N)
    V = omega_v @ X + beta_v @ ones_row  # (D_h, N)

    E = (Q.T @ K) / np.sqrt(D_h)  # (N, N)
    A = softmax_rows(E)           # (N, N)
    X_head = V @ A.T              # (D_h, N)
    return X_head, A


def multihead_scaled_self_attention(X,
                                   omega_q1, omega_k1, omega_v1, beta_q1, beta_k1, beta_v1,
                                   omega_q2, omega_k2, omega_v2, beta_q2, beta_k2, beta_v2,
                                   omega_c):
    X1, A1 = single_head_scaled_attention(X, omega_q1, omega_k1, omega_v1, beta_q1, beta_k1, beta_v1)
    X2, A2 = single_head_scaled_attention(X, omega_q2, omega_k2, omega_v2, beta_q2, beta_k2, beta_v2)

    # Конкатенация по признаковому измерению: (2 * D_h, N) == (D, N)
    X_concat = np.vstack([X1, X2])
    X_prime = omega_c @ X_concat  # (D, N)
    return X_prime, (A1, A2)


X_prime, (A1, A2) = multihead_scaled_self_attention(
    X,
    omega_q1, omega_k1, omega_v1, beta_q1, beta_k1, beta_v1,
    omega_q2, omega_k2, omega_v2, beta_q2, beta_k2, beta_v2,
    omega_c,
)

print("Форма результата X_prime:", X_prime.shape)
print("Формы матриц вниманий голов:", A1.shape, A2.shape)

### Задача 2. Визуализация матриц вниманий голов

**Постановка задачи.**  
Визуализировать матрицы вниманий двух голов с помощью тепловых карт
и сравнить, как разные головы распределяют внимание по позициям.

**Теоретический минимум.**  
Строка $n$ матрицы $A^{(h)}$ показывает, на какие позиции входа «смотрит»
элемент с индексом $n$ через голову $h$.

Гипотеза: даже при случайной инициализации разные головы могут фокусироваться
на разных паттернах.

In [ ]:
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(A1, cmap="viridis")
plt.title("Голова 1")
plt.xlabel("m (ключи)")
plt.ylabel("n (запросы)")
plt.colorbar(label="коэффициент внимания")

plt.subplot(1, 2, 2)
plt.imshow(A2, cmap="viridis")
plt.title("Голова 2")
plt.xlabel("m (ключи)")
plt.ylabel("n (запросы)")
plt.colorbar(label="коэффициент внимания")

plt.tight_layout()
plt.show()

### Задача 3. Подготовка языковой модели для экспериментов с декодированием

**Постановка задачи.**  
Загрузить предобученную языковую модель из библиотеки `transformers`
(возьмём компактную `distilgpt2`) и подготовить функцию для получения
распределения вероятностей по следующему токену.

**Теоретический минимум.**  
Модель возвращает логиты по всем токенам словаря; softmax по ним даёт
распределение $\boldsymbol{\pi}$ над токенами.

In [ ]:
!pip -q install transformers

from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed
import torch
import torch.nn.functional as F

model_name = "distilgpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

model.eval()

set_seed(0)

print("Размер словаря:", tokenizer.vocab_size)

### Задача 4. Реализация базовых стратегий декодирования

**Постановка задачи.**  
Реализовать функции, которые по текущему контексту возвращают индекс
следующего токена по одной из стратегий:

- случайное сэмплирование по всему распределению,
- жадный выбор (argmax).

**Теоретический минимум.**  

Пусть $\boldsymbol{\pi} \in \mathbb{R}^V$ — распределение вероятностей
по словарю размера $V$. Тогда:

- случайное сэмплирование: $x \sim \mathrm{Categorical}(\boldsymbol{\pi})$;
- жадный выбор: $x = \arg\max_i \pi_i$.

In [ ]:
import numpy as np


def get_next_token_distribution(input_tokens, model):
    """Возвращает распределение вероятностей по следующему токену."""
    with torch.no_grad():
        outputs = model(**input_tokens)
        logits = outputs.logits  # (batch, seq_len, vocab_size)
        last_logits = logits[0, -1, :]  # (vocab_size,)
        probs = F.softmax(last_logits, dim=-1).cpu().numpy()
    return probs


def sample_random(probs: np.ndarray) -> int:
    """Случайное сэмплирование по полному распределению."""
    return int(np.random.choice(len(probs), p=probs))


def sample_greedy(probs: np.ndarray) -> int:
    """Жадный выбор (argmax)."""
    return int(np.argmax(probs))

# Короткая проверка
prompt = "The best thing about Bath is"
input_tokens = tokenizer(prompt, return_tensors="pt")
probs_example = get_next_token_distribution(input_tokens, model)

print("Сумма вероятностей:", probs_example.sum())
print("Случайный токен:", sample_random(probs_example))
print("Жадный токен:", sample_greedy(probs_example))

### Задача 5. Реализация top‑k и nucleus (top‑p) sampling

**Постановка задачи.**  
Реализовать стратегии:

- top‑k sampling: сэмплировать только среди k наиболее вероятных токенов;
- nucleus sampling (top‑p): сэмплировать из минимального множества токенов,
  чья суммарная вероятность ≥ p.

**Теоретический минимум.**  

Top‑k:

1. Найти k наибольших вероятностей.
2. Обнулить остальные.
3. Нормировать и сэмплировать.

Top‑p:

1. Отсортировать вероятности по убыванию.
2. Взять минимальное число токенов, дающее суммарную вероятность ≥ p.
3. Обнулить остальные, нормировать и сэмплировать.

In [ ]:
def sample_top_k(probs: np.ndarray, k: int = 20) -> int:
    """Top-k sampling: сэмплирование среди k наиболее вероятных токенов."""
    probs = probs.copy()
    sorted_indices = np.argsort(probs)[::-1]
    top_k_indices = sorted_indices[:k]

    mask = np.zeros_like(probs, dtype=bool)
    mask[top_k_indices] = True

    filtered_probs = probs * mask
    filtered_probs /= filtered_probs.sum()

    return int(np.random.choice(len(filtered_probs), p=filtered_probs))


def sample_top_p(probs: np.ndarray, p: float = 0.9) -> int:
    """Nucleus sampling (top-p)."""
    probs = probs.copy()
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]

    cumulative = np.cumsum(sorted_probs)
    cutoff_idx = np.searchsorted(cumulative, p) + 1
    nucleus_indices = sorted_indices[:cutoff_idx]

    mask = np.zeros_like(probs, dtype=bool)
    mask[nucleus_indices] = True

    filtered_probs = probs * mask
    filtered_probs /= filtered_probs.sum()

    return int(np.random.choice(len(filtered_probs), p=filtered_probs))

# Проверка
print("Top-k токен:", sample_top_k(probs_example, k=10))
print("Top-p токен:", sample_top_p(probs_example, p=0.9))

### Задача 6. Генерация текста и сравнение стратегий

**Постановка задачи.**  
Написать функцию генерации текста, которая на каждом шаге использует
одну из стратегий выбора токена. Сравнить результаты по нескольким
начальным фразам и фиксированным семенам случайности.

**Гипотеза.**  
Жадная стратегия даст наиболее детерминированный и предсказуемый текст,
а стохастические стратегии приведут к большему разнообразию и, возможно,
к неожиданным продолжениям.

In [ ]:
STRATEGIES = {
    "random": sample_random,
    "greedy": sample_greedy,
    "top_k": sample_top_k,
    "top_p": sample_top_p,
}


def generate_text(prompt: str,
                  model,
                  tokenizer,
                  strategy: str = "greedy",
                  max_new_tokens: int = 20,
                  top_k: int = 20,
                  top_p: float = 0.9,
                  seed: int = 0) -> str:
    """Генерация текста с заданной стратегией выбора следующего токена."""
    np.random.seed(seed)
    set_seed(seed)

    input_tokens = tokenizer(prompt, return_tensors="pt")

    for _ in range(max_new_tokens):
        probs = get_next_token_distribution(input_tokens, model)

        if strategy == "random":
            next_id = sample_random(probs)
        elif strategy == "greedy":
            next_id = sample_greedy(probs)
        elif strategy == "top_k":
            next_id = sample_top_k(probs, k=top_k)
        elif strategy == "top_p":
            next_id = sample_top_p(probs, p=top_p)
        else:
            raise ValueError(f"Неизвестная стратегия: {strategy}")

        next_token = torch.tensor([[next_id]])
        input_tokens["input_ids"] = torch.cat(
            [input_tokens["input_ids"], next_token], dim=1
        )
        input_tokens["attention_mask"] = torch.cat(
            [
                input_tokens["attention_mask"],
                torch.ones((1, 1), dtype=input_tokens["attention_mask"].dtype),
            ],
            dim=1,
        )

    generated = tokenizer.decode(input_tokens["input_ids"][0], skip_special_tokens=True)
    return generated

prompt = "The best thing about Bath is"

for strategy in ["random", "greedy", "top_k", "top_p"]:
    text = generate_text(prompt, model, tokenizer,
                         strategy=strategy,
                         max_new_tokens=20,
                         top_k=10,
                         top_p=0.9,
                         seed=0)
    print("\nСтратегия:", strategy)
    print(text)

## 5. Интерпретация результатов

- Многоголовое самовнимание расширяет базовый механизм самовнимания:
  разные головы формируют разные карты внимания и, соответственно,
  разные «ракурсы» на одну и ту же последовательность.
- Масштабирование скалярных произведений на $\sqrt{D_h}$ смягчает softmax
  и предотвращает чрезмерную концентрацию внимания на одной позиции.
- В генеративной задаче стратегии декодирования сильно влияют на характер
  текста: от детерминированного и шаблонного (greedy) до более
  разнообразного и непредсказуемого (random, top‑k, top‑p).
- Параметры k и p играют роль «регуляторов хаоса»: чрезмерно большие значения
  приводят к деградации качества, малые — к почти жадному поведению.

## 6. Выводы

1. Многоголовое самовнимание позволяет разным головам специализироваться
   на различных типах зависимостей и признаков в последовательности.
2. Матричная реализация многоголового самовнимания подчиняется
   повторяющемуся шаблону: $X \to Q, K, V \to$ матрицы вниманий $\to$
   взвешивание значений $\to$ объединение голов.
3. В генеративных моделях выбор стратегии декодирования является неотъемлемой
   частью дизайна системы и влияет не меньше, чем сама архитектура модели.
4. Top‑k и top‑p предоставляют удобные ручки для настройки компромисса
   между качеством и разнообразием текста.
5. Понимание связи между вероятностным распределением по токенам и
   стратегией выбора следующего токена формирует аналитическое и
   исследовательское мышление при работе с трансформерами.

## 7. Самостоятельные задачи (с эталонными идеями решений)

### Задача 7.1. Влияние числа голов

**Задача.**  
Модифицируйте реализацию многоголового самовнимания так, чтобы число голов
$H$ было параметром функции. Проведите эксперименты с $H = 1, 2, 4$
при фиксированном $D$ и сравните получающиеся матрицы вниманий.

**Идея решения.**  
Хранить параметры голов в списке и итерироваться по ним:



Ожидается, что при увеличении числа голов паттерны внимания становятся
более разнообразными.

---

### Задача 7.2. Сравнение стратегий декодирования по среднему лог‑правдоподобию

**Задача.**  
Для фиксированного промпта и набора стратегий декодирования сгенерируйте
несколько продолжений текста. Для каждого продолжения оцените среднее
лог‑правдоподобие токенов под исходной моделью и сравните стратегии.

**Идея решения.**  

1. Сгенерировать текст стратегией $s$.
2. Прогнать последовательность через модель и посчитать
   $\frac{1}{T} \sum_t \log p(x_t \mid x_{<t})$.
3. Усреднить по нескольким запускам.



---

### Задача 7.3. Анализ повторов в сгенерированном тексте

**Задача.**  
Для разных стратегий декодирования соберите несколько сгенерированных
текстов и проанализируйте частоту повторения n‑грамм (например, биграмм
и триграмм). Сравните долю уникальных n‑грамм между стратегиями.

**Идея решения.**  
Реализовать функцию, которая по тексту строит список n‑грамм, считает их
частоты с помощью `dict` и возвращает число уникальных и общее число n‑грамм.
Ожидается, что более стохастические стратегии дают больший процент
уникальных n‑грамм, но чаще порождают менее связный текст.